# Crayon Commotion — Generate Your Materials

This notebook turns one of your own images into a printable Crayon Commotion activity kit: a blank student sheet, per-mini-page encoding instructions, a wall-sized color reference chart, and a teacher's cheat sheet.

**You don't need to know Python to use this.** Run each cell from top to bottom by clicking the play button on its left (or use *Runtime → Run all*), and follow the prompts.

No install, no account beyond the one you're already signed into for Colab.

## Step 0: Setup

Downloads the Crayon Commotion code and installs the two packages it needs. This only has to run once per session and takes a few seconds.

In [ ]:
# If this activity ends up published at a different repo/path than
# Ready-Remix-Run/2026's crayon-commotion/ folder, update the two paths below to match.
!git clone -q https://github.com/Ready-Remix-Run/2026.git
!pip install -q pillow reportlab

import sys
sys.path.insert(0, "2026/crayon-commotion/src")

print("Setup complete.")

## Step 1: Upload your image

Any PNG or JPG works. Photos with large solid-color areas (logos, cartoons) will tend to produce more mini-pages that are entirely one color — see the `ensure_variety` option in Step 2 if that happens.

In [ ]:
from google.colab import files

print("Choose an image file to upload (PNG or JPG).")
uploaded = files.upload()
uploaded_image_path = next(iter(uploaded))

print(f"\nUploaded '{uploaded_image_path}'")

## Step 2: Choose your settings

- **palette** — which box of crayons the colors are matched against. `Crayola 24` gives more color accuracy; `Crayola 16` is the standard smaller box, if that's what you have on hand.
- **encoding** — how each square's color is written on the encoding instructions.
  - `Decimal` — a plain number (e.g. `18`). The simplest option.
  - `Binary` — a fixed-width binary number (e.g. `10010`). Participants decode it to decimal first, then look up that decimal number on the color reference chart (which stays decimal-keyed).
  - `RGB Binary` — three 8-bit binary numbers stacked in the cell, one per red/green/blue component (e.g. `11101110` / `00100000` / `01001101`). Participants decode all three to decimal and look up that exact `R,G,B` triple on the reference chart, which switches to showing RGB values instead of palette ids. This doesn't use a lookup-table id at all -- it's the actual color, more work to decode, better for advanced CS-teacher audiences.
  - `RGB Hex` — the same idea as RGB Binary, but each component is a 2-digit hex byte instead (e.g. `FF` / `A2` / `00`). Shorter and easier to read than RGB Binary, still the actual color rather than a palette id, and the reference chart shows the same RGB decimal triples as RGB Binary.
- **block_width_px / block_height_px** — the size of one student's mini-page, in pixels. Each pixel becomes one colored square. `6 x 4` (24 squares) is a good starting point. The RGB encodings need 3 lines of text per square, so consider a larger `cell_size_cm` for legibility (RGB Binary especially).
- **blocks_wide / blocks_tall** — how many mini-pages make up the full picture, across and down. Your image gets resized to `blocks_wide × block_width_px` by `blocks_tall × block_height_px` pixels, so these two settings together control both the final resolution and exactly how many mini-pages get created (`blocks_wide × blocks_tall`). A very different aspect ratio from your original image will visibly stretch or squash it — if that happens, adjust these two numbers to better match your image's proportions.
- **cell_size_cm** — the printed size of one square on the student sheets, in centimeters.
- **cheat_sheet_cell_size_cm** — same, but for the teacher's cheat sheet. Smaller means fewer pages and less color-printing cost.
- **ensure_variety** — if your image has large flat/solid areas, turn this on to guarantee no student gets a mini-page that's entirely one color. Leave it off for photos, which rarely need it.

Edit the values below, then run this cell.

In [ ]:
title = "My Crayon Commotion Image"  #@param {type:"string"}

palette_choice = "Crayola 24"  #@param ["Crayola 24", "Crayola 16"]
encoding = "Binary"  #@param ["Decimal", "Binary", "RGB Binary", "RGB Hex"]

block_width_px = 6  #@param {type:"integer"}
block_height_px = 4  #@param {type:"integer"}

blocks_wide = 20  #@param {type:"integer"}
blocks_tall = 15  #@param {type:"integer"}

cell_size_cm = 1.0  #@param {type:"number"}
cheat_sheet_cell_size_cm = 0.5  #@param {type:"number"}

ensure_variety = False  #@param {type:"boolean"}

print(f"Final image size: {blocks_wide * block_width_px} x {blocks_tall * block_height_px} px")
print(f"Mini-pages: {blocks_wide * blocks_tall} (each {block_width_px} x {block_height_px} px)")
print(f"Palette: {palette_choice}")
print(f"Encoding: {encoding}")

## Step 3: Generate your materials

This resizes your image, matches every pixel to the nearest crayon color, builds the mini-pages, and renders all four PDFs. When it finishes, your browser will download a zip file containing everything.

In [ ]:
import zipfile
from pathlib import Path

from commotion.palettes.loader import load_palette
from commotion.imaging.resize import load_and_resize_image
from commotion.imaging.quantize import quantize_image
from commotion.imaging.variety import count_monochrome_blocks, ensure_block_variety
from commotion.encoders.decimal import DecimalEncoder
from commotion.encoders.binary import BinaryEncoder, bits_needed_for_palette
from commotion.encoders.rgb_binary import RGBBinaryEncoder
from commotion.encoders.hexadecimal import RGBHexEncoder
from commotion.encoders.grid import encode_grid
from commotion.worksheets.builder import build_worksheet
from commotion.worksheets.splitter import split_into_student_sheets
from commotion.renderers.pdf import (
    render_blank_block_sheet,
    render_encoding_sheets,
    render_reference_sheet,
    render_cheat_sheet,
)
from commotion.models import DimensionOption

output_dir = Path("/content/output")
output_dir.mkdir(exist_ok=True)

palette_files = {
    "Crayola 24": "crayola24.csv",
    "Crayola 16": "crayola16.csv",
}
palette_path = f"2026/crayon-commotion/palettes/{palette_files[palette_choice]}"
palette = load_palette(palette_path, palette_name=palette_choice)

target_width = blocks_wide * block_width_px
target_height = blocks_tall * block_height_px

pixels = load_and_resize_image(uploaded_image_path, rows=target_height, cols=target_width)
grid = quantize_image(pixels, palette)

mono_before, total_blocks = count_monochrome_blocks(grid, block_height_px, block_width_px)
print(f"Monochrome mini-pages before variety pass: {mono_before}/{total_blocks}")

if ensure_variety:
    grid = ensure_block_variety(grid, palette, block_height_px, block_width_px)
    mono_after, _ = count_monochrome_blocks(grid, block_height_px, block_width_px)
    print(f"Monochrome mini-pages after variety pass:  {mono_after}/{total_blocks}")

if encoding == "Binary":
    encoder = BinaryEncoder(bits_needed_for_palette(palette))
    reference_label_fn = lambda color: str(color.id)
elif encoding == "RGB Binary":
    encoder = RGBBinaryEncoder()
    reference_label_fn = lambda color: f"{color.r},{color.g},{color.b}"
elif encoding == "RGB Hex":
    encoder = RGBHexEncoder()
    reference_label_fn = lambda color: f"{color.r},{color.g},{color.b}"
else:
    encoder = DecimalEncoder()
    reference_label_fn = lambda color: str(color.id)

encoded = encode_grid(grid, encoder)
worksheet = build_worksheet(
    title=title,
    cells=encoded,
    encoder_name=encoding,
    palette_name=palette.name,
)
print(f"Encoding: {encoding} (e.g. top-left cell = '{encoded[0][0].value}')")

sheet_size = DimensionOption(rows=block_height_px, cols=block_width_px)
sheets = split_into_student_sheets(worksheet, sheet_size)
print(f"Split into {len(sheets)} mini-pages of {block_width_px} x {block_height_px} px each")

blank_path = output_dir / "blank_sheet.pdf"
encoding_path = output_dir / "encoding_sheets.pdf"
reference_path = output_dir / "reference_sheet.pdf"
cheat_path = output_dir / "cheat_sheet.pdf"

render_blank_block_sheet(sheet_size.rows, sheet_size.cols, blank_path, cell_size_cm=cell_size_cm)
render_encoding_sheets(sheets, encoding_path, cell_size_cm=cell_size_cm)
render_reference_sheet(palette, reference_path, label_fn=reference_label_fn)
render_cheat_sheet(
    worksheet, cheat_path,
    block_rows=sheet_size.rows, block_cols=sheet_size.cols,
    cell_size_cm=cheat_sheet_cell_size_cm,
)

zip_path = Path("/content/crayon_commotion_materials.zip")
with zipfile.ZipFile(zip_path, "w") as zf:
    for pdf_path in [blank_path, encoding_path, reference_path, cheat_path]:
        zf.write(pdf_path, arcname=pdf_path.name)

print(f"\nDone! Downloading {zip_path.name}...")

from google.colab import files as colab_files
colab_files.download(str(zip_path))